# Clase 9 — Visualización y Dashboards para el Sector Pesquero
## Curso: Inteligencia Artificial Aplicada a la Producción Pesquera
### UTN FRCh · PesquerosEnIA · 2026

**Docente:** Damian Adolfo Giacone  
**Notebook de apoyo:** Ariel Giamportone

---

## Objetivos de este notebook

1. Construir visualizaciones interactivas con **Plotly** para datos pesqueros
2. Diseñar un **dashboard operativo** de flota pesquera con KPIs clave
3. Crear un **mapa interactivo** de actividad de flota con Folium
4. Entender la diferencia entre herramientas de BI (Power BI, Tableau) y Python
5. Exportar visualizaciones listas para presentar a gerencias y organismos

---

## ¿Por qué importan los dashboards en el sector pesquero?

Un capitán de flota gestiona simultáneamente:
- 10-30 buques en mar con datos en tiempo real
- Cuotas por especie con seguimiento semanal
- Costos de combustible por marea
- Rendimiento (kg/día en mar) por barco y zona
- Cumplimiento regulatorio (áreas de veda, límites de captura)

Un buen dashboard convierte **datos crudos** en **decisiones en minutos**.

## Parte 0 — Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Intentamos importar Plotly (interactivo)
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY = True
    print('Plotly disponible — se usarán gráficos interactivos')
except ImportError:
    PLOTLY = False
    print('Plotly no instalado — se usan matplotlib (estatico)')
    print('Para instalar: pip install plotly')

# Intentamos Folium (mapas)
try:
    import folium
    from folium.plugins import HeatMap, MarkerCluster
    FOLIUM = True
    print('Folium disponible — se generarán mapas interactivos')
except ImportError:
    FOLIUM = False
    print('Folium no instalado — se usa matplotlib para mapas')
    print('Para instalar: pip install folium')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 13})

UTNAZUL    = '#00467F'
UTNCELESTE = '#0099CC'
PESCATEAL  = '#059669'
TECHGOLD   = '#D4A017'
PESCAOCEAN = '#3B82F6'
ARIELBLUE  = '#2C5F7C'
ROJO       = '#E74C3C'
GRIS       = '#6B7280'

---
## Parte 1 — Dataset: Operaciones de flota pesquera

Generamos un dataset realista de operaciones de una flota de 15 arrastreros de altura durante 6 meses.

In [ ]:
def generar_dataset_flota(n_barcos=15, n_meses=6, seed=42):
    """
    Genera dataset de operaciones de flota pesquera.
    Incluye mareas completas con datos de captura, combustible y eficiencia.
    """
    rng = np.random.default_rng(seed)
    
    barcos = [
        {'id': f'BP-{i+101:03d}', 'nombre': nombre, 'eslora_m': eslora,
         'potencia_hp': pot, 'base': base}
        for i, (nombre, eslora, pot, base) in enumerate([
            ('Patagónico I',    42, 1200, 'Comodoro Rivadavia'),
            ('Patagónico II',   44, 1350, 'Comodoro Rivadavia'),
            ('Mar del Plata A', 38, 1050, 'Mar del Plata'),
            ('Mar del Plata B', 40, 1100, 'Mar del Plata'),
            ('San Jorge I',     46, 1400, 'Rawson'),
            ('San Jorge II',    45, 1380, 'Rawson'),
            ('Madrynero',       35, 950,  'Puerto Madryn'),
            ('Fueguino I',      50, 1600, 'Ushuaia'),
            ('Fueguino II',     48, 1550, 'Ushuaia'),
            ('Rioplatense',     36, 980,  'Mar del Plata'),
            ('Atlantis',        43, 1250, 'Comodoro Rivadavia'),
            ('Merlusero',       41, 1150, 'Rawson'),
            ('Calamarón',       39, 1080, 'Mar del Plata'),
            ('Austral Sur',     47, 1500, 'Ushuaia'),
            ('Norpatagónico',   44, 1300, 'Puerto Madryn'),
        ][:n_barcos])
    ]
    
    fecha_inicio = datetime(2025, 7, 1)
    registros = []
    
    especies = ['Merluza hubbsi', 'Calamar illex', 'Langostino patagónico', 'Polaca', 'Castañeta']
    zonas = [
        {'nombre': 'Zona A - Norte PCA', 'lat_c': -40.5, 'lon_c': -58.0},
        {'nombre': 'Zona B - Golfo San Jorge', 'lat_c': -45.5, 'lon_c': -61.5},
        {'nombre': 'Zona C - Banco Burdwood', 'lat_c': -52.5, 'lon_c': -60.0},
        {'nombre': 'Zona D - Frente Malvinas', 'lat_c': -44.0, 'lon_c': -58.5},
        {'nombre': 'Zona E - Sur Patagónico', 'lat_c': -48.0, 'lon_c': -63.0},
    ]
    
    marea_id = 1
    for barco in barcos:
        fecha_actual = fecha_inicio
        fecha_fin = fecha_inicio + timedelta(days=30 * n_meses)
        
        # Factor de eficiencia base del barco (algunos son más eficientes)
        efic_barco = rng.uniform(0.85, 1.15)
        
        while fecha_actual < fecha_fin:
            # Duración de la marea: 10-22 días
            duracion_dias = int(rng.integers(10, 22))
            
            # Selección de especie principal (estacional)
            mes = fecha_actual.month % 12
            if mes in [7, 8, 9]:    # invierno: merluza domina
                esp_probs = [0.55, 0.25, 0.08, 0.08, 0.04]
            elif mes in [10, 11, 12]:  # primavera: calamar y langostino
                esp_probs = [0.30, 0.35, 0.20, 0.10, 0.05]
            else:  # verano/otoño
                esp_probs = [0.40, 0.20, 0.25, 0.10, 0.05]
            
            especie = rng.choice(especies, p=esp_probs)
            zona = rng.choice(zonas)
            
            # Posición aproximada en la zona
            lat = zona['lat_c'] + rng.uniform(-1.5, 1.5)
            lon = zona['lon_c'] + rng.uniform(-2.0, 2.0)
            
            # Capturas: kg/día base según especie y zona
            base_kg_dia = {
                'Merluza hubbsi': 8500, 'Calamar illex': 12000,
                'Langostino patagónico': 4500, 'Polaca': 7000, 'Castañeta': 5500
            }[especie]
            
            kg_dia = base_kg_dia * efic_barco * rng.uniform(0.75, 1.30)
            captura_ton = kg_dia * duracion_dias / 1000
            
            # Combustible
            consumo_base = barco['potencia_hp'] * 0.18  # litros/hora base
            consumo_total_l = consumo_base * 24 * duracion_dias * rng.uniform(0.85, 1.10)
            precio_gasoil = 0.85  # USD/litro
            costo_combustible = consumo_total_l * precio_gasoil
            
            # Eficiencia
            eficiencia = captura_ton / (consumo_total_l / 1000)  # ton / (kl combustible)
            
            # Precio del producto
            precio_usd_ton = {
                'Merluza hubbsi': 1800, 'Calamar illex': 1500,
                'Langostino patagónico': 4500, 'Polaca': 1200, 'Castañeta': 1100
            }[especie]
            
            ingreso_usd = captura_ton * precio_usd_ton * rng.uniform(0.90, 1.10)
            margen = (ingreso_usd - costo_combustible) / ingreso_usd
            
            registros.append({
                'marea_id': f'M{marea_id:04d}',
                'barco_id': barco['id'],
                'barco_nombre': barco['nombre'],
                'base_puerto': barco['base'],
                'fecha_salida': fecha_actual,
                'fecha_regreso': fecha_actual + timedelta(days=duracion_dias),
                'mes': fecha_actual.strftime('%Y-%m'),
                'duracion_dias': duracion_dias,
                'zona': zona['nombre'],
                'lat_promedio': round(lat, 4),
                'lon_promedio': round(lon, 4),
                'especie_principal': especie,
                'captura_ton': round(captura_ton, 1),
                'combustible_kl': round(consumo_total_l / 1000, 2),
                'costo_combustible_usd': round(costo_combustible, 0),
                'precio_usd_ton': precio_usd_ton,
                'ingreso_usd': round(ingreso_usd, 0),
                'margen_pct': round(margen * 100, 1),
                'eficiencia_ton_kl': round(eficiencia, 2),
                'ton_por_dia': round(captura_ton / duracion_dias, 2),
            })
            
            # Período en puerto entre mareas (3-8 días)
            dias_puerto = int(rng.integers(3, 8))
            fecha_actual = fecha_actual + timedelta(days=duracion_dias + dias_puerto)
            marea_id += 1
    
    df = pd.DataFrame(registros)
    df['fecha_salida'] = pd.to_datetime(df['fecha_salida'])
    df['fecha_regreso'] = pd.to_datetime(df['fecha_regreso'])
    return df


df = generar_dataset_flota(n_barcos=15, n_meses=6)

print(f'Dataset de flota generado:')
print(f'  Mareas totales:    {len(df)}')
print(f'  Barcos:            {df["barco_id"].nunique()}')
print(f'  Período:           {df["fecha_salida"].min().date()} → {df["fecha_regreso"].max().date()}')
print(f'  Captura total:     {df["captura_ton"].sum():,.0f} toneladas')
print(f'  Ingreso total:     USD {df["ingreso_usd"].sum():,.0f}')
print(f'  Combustible total: {df["combustible_kl"].sum():,.0f} kL')
df.head(3)

---
## Parte 2 — KPIs del Dashboard Operativo

In [ ]:
# Calcular KPIs globales del período
kpis = {
    'Total mareas': len(df),
    'Captura total (ton)': f"{df['captura_ton'].sum():,.0f}",
    'Ingreso total (MUSD)': f"{df['ingreso_usd'].sum()/1e6:.1f}",
    'Costo combustible (MUSD)': f"{df['costo_combustible_usd'].sum()/1e6:.1f}",
    'Eficiencia media (ton/kL)': f"{df['eficiencia_ton_kl'].mean():.2f}",
    'Margen promedio (%)': f"{df['margen_pct'].mean():.1f}%",
    'Ton/día promedio': f"{df['ton_por_dia'].mean():.1f}",
    'Duración media marea (días)': f"{df['duracion_dias'].mean():.1f}",
}

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
axes = axes.flatten()

kpi_colors = [UTNAZUL, PESCATEAL, TECHGOLD, ROJO,
              PESCAOCEAN, ARIELBLUE, UTNCELESTE, GRIS]

for ax, (nombre, valor), color in zip(axes, kpis.items(), kpi_colors):
    ax.set_facecolor(color)
    ax.text(0.5, 0.62, str(valor), transform=ax.transAxes,
            ha='center', va='center', fontsize=22, fontweight='bold',
            color='white')
    ax.text(0.5, 0.22, nombre, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color='white',
            fontweight='normal', wrap=True)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('white')
        spine.set_linewidth(2)

plt.suptitle('Dashboard Operativo — Flota Pesquera Patagónica (Jul 2025 – Dic 2025)',
             fontsize=13, fontweight='bold', color=UTNAZUL, y=1.02)
plt.tight_layout()
plt.show()

---
## Parte 3 — Análisis temporal: producción y eficiencia por mes

In [ ]:
df_mes = df.groupby('mes').agg(
    captura_ton=('captura_ton', 'sum'),
    ingreso_usd=('ingreso_usd', 'sum'),
    costo_combustible=('costo_combustible_usd', 'sum'),
    n_mareas=('marea_id', 'count'),
    eficiencia_media=('eficiencia_ton_kl', 'mean'),
    margen_medio=('margen_pct', 'mean')
).reset_index()

meses_labels = ['Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Captura mensual
ax = axes[0, 0]
bars = ax.bar(meses_labels, df_mes['captura_ton'],
              color=[UTNAZUL if v >= df_mes['captura_ton'].mean() else UTNCELESTE
                     for v in df_mes['captura_ton']], alpha=0.85)
ax.axhline(df_mes['captura_ton'].mean(), color=TECHGOLD, linestyle='--', lw=2, label='Promedio')
ax.set_ylabel('Toneladas')
ax.set_title('Captura mensual total (15 barcos)', fontweight='bold')
ax.legend()
for bar, val in zip(bars, df_mes['captura_ton']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,.0f}t', ha='center', fontsize=9, fontweight='bold')

# 2. Ingreso vs Costo combustible
ax = axes[0, 1]
x = np.arange(len(meses_labels))
w = 0.38
ax.bar(x - w/2, df_mes['ingreso_usd']/1000, w, label='Ingreso', color=PESCATEAL, alpha=0.85)
ax.bar(x + w/2, df_mes['costo_combustible']/1000, w, label='Costo gasoil', color=ROJO, alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(meses_labels)
ax.set_ylabel('Miles de USD')
ax.set_title('Ingresos vs Costo de combustible', fontweight='bold')
ax.legend()

# 3. Eficiencia por mes (ton/kL)
ax = axes[1, 0]
ax.plot(meses_labels, df_mes['eficiencia_media'], 'o-',
        color=ARIELBLUE, linewidth=2.5, markersize=8, markerfacecolor=TECHGOLD)
ax.fill_between(range(len(meses_labels)), df_mes['eficiencia_media'],
                alpha=0.15, color=ARIELBLUE)
ax.set_ylabel('Ton / kL de combustible')
ax.set_title('Eficiencia operativa mensual', fontweight='bold')
ax.set_ylim(df_mes['eficiencia_media'].min() * 0.95,
             df_mes['eficiencia_media'].max() * 1.05)
for i, val in enumerate(df_mes['eficiencia_media']):
    ax.annotate(f'{val:.2f}', (i, val), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=10, fontweight='bold')

# 4. Margen % por mes
ax = axes[1, 1]
colors_margen = [PESCATEAL if m > 50 else UTNCELESTE if m > 35 else ROJO
                 for m in df_mes['margen_medio']]
bars = ax.bar(meses_labels, df_mes['margen_medio'], color=colors_margen, alpha=0.85)
ax.axhline(50, color=PESCATEAL, linestyle='--', lw=1.5, alpha=0.7, label='>50%: excelente')
ax.axhline(35, color=UTNCELESTE, linestyle='--', lw=1.5, alpha=0.7, label='>35%: bueno')
ax.set_ylabel('Margen (%)')
ax.set_title('Margen operativo mensual (%)', fontweight='bold')
ax.legend(fontsize=9)
for bar, val in zip(bars, df_mes['margen_medio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Análisis Temporal — Operaciones de Flota · Jul-Dic 2025',
             fontsize=13, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

---
## Parte 4 — Ranking de barcos: eficiencia y rendimiento

In [ ]:
df_barco = df.groupby(['barco_id', 'barco_nombre', 'base_puerto']).agg(
    n_mareas=('marea_id', 'count'),
    captura_total_ton=('captura_ton', 'sum'),
    ingreso_total_usd=('ingreso_usd', 'sum'),
    costo_combustible_usd=('costo_combustible_usd', 'sum'),
    eficiencia_media=('eficiencia_ton_kl', 'mean'),
    margen_medio=('margen_pct', 'mean'),
    ton_dia_media=('ton_por_dia', 'mean'),
).reset_index()

df_barco['utilidad_usd'] = df_barco['ingreso_total_usd'] - df_barco['costo_combustible_usd']
df_barco = df_barco.sort_values('utilidad_usd', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Ranking por utilidad
ax = axes[0]
colors = [PESCATEAL if i < 5 else UTNCELESTE if i < 10 else ROJO
          for i in range(len(df_barco))]
bars = ax.barh(df_barco['barco_nombre'][::-1], df_barco['utilidad_usd'][::-1] / 1000,
               color=colors[::-1], alpha=0.85)
ax.set_xlabel('Utilidad (Miles USD, 6 meses)')
ax.set_title('Ranking de barcos por utilidad generada\n(Ingreso − Costo de combustible)',
             fontweight='bold', color=UTNAZUL)
for bar, val in zip(bars, df_barco['utilidad_usd'][::-1] / 1000):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f'USD {val:.0f}K', va='center', fontsize=9)

legend_patches = [
    mpatches.Patch(color=PESCATEAL, label='Top 5'),
    mpatches.Patch(color=UTNCELESTE, label='Top 6-10'),
    mpatches.Patch(color=ROJO, label='Bottom 5'),
]
ax.legend(handles=legend_patches, loc='lower right')

# Scatter eficiencia vs margen
ax = axes[1]
sc = ax.scatter(
    df_barco['eficiencia_media'],
    df_barco['margen_medio'],
    s=df_barco['captura_total_ton'] / 5,
    c=df_barco['utilidad_usd'],
    cmap='RdYlGn', alpha=0.8, edgecolors='white', linewidth=1
)
plt.colorbar(sc, ax=ax, label='Utilidad USD')

for _, row in df_barco.iterrows():
    ax.annotate(row['barco_nombre'].split()[0],
                (row['eficiencia_media'], row['margen_medio']),
                textcoords='offset points', xytext=(5, 3), fontsize=8, color=UTNAZUL)

ax.axvline(df_barco['eficiencia_media'].mean(), color=GRIS, linestyle='--', alpha=0.5)
ax.axhline(df_barco['margen_medio'].mean(), color=GRIS, linestyle='--', alpha=0.5)
ax.set_xlabel('Eficiencia (ton / kL combustible)')
ax.set_ylabel('Margen operativo (%)')
ax.set_title('Eficiencia vs Margen por barco\n(tamaño del punto = captura total)',
             fontweight='bold', color=UTNAZUL)

# Cuadrantes
xlim, ylim = ax.get_xlim(), ax.get_ylim()
xm, ym = df_barco['eficiencia_media'].mean(), df_barco['margen_medio'].mean()
ax.text((xlim[0]+xm)/2, ym + 1, 'Alta margen\nBaja eficiencia', fontsize=8,
        ha='center', color=UTNCELESTE, alpha=0.6)
ax.text((xlim[1]+xm)/2, ym + 1, '✓ ZONA IDEAL', fontsize=9,
        ha='center', color=PESCATEAL, fontweight='bold', alpha=0.8)

plt.suptitle('Performance Individual de Barcos — 6 meses de operación',
             fontsize=13, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

---
## Parte 5 — Mapa de actividad de flota

Si tenés Folium instalado, se genera un mapa interactivo. Si no, se usa matplotlib.

In [ ]:
if FOLIUM:
    # Mapa interactivo con Folium
    m = folium.Map(location=[-45, -61], zoom_start=5,
                   tiles='CartoDB dark_matter')
    
    # Heatmap de actividad
    heat_data = [[row['lat_promedio'], row['lon_promedio'], row['captura_ton']]
                 for _, row in df.iterrows()]
    HeatMap(heat_data, radius=15, blur=12, min_opacity=0.4,
            gradient={'0.3': 'blue', '0.6': '#0099CC', '0.9': '#059669', '1.0': 'yellow'}).add_to(m)
    
    # Markers de puertos
    puertos = {
        'Mar del Plata': (-38.00, -57.53),
        'Rawson': (-43.30, -65.10),
        'Puerto Madryn': (-42.77, -65.03),
        'Comodoro Rivadavia': (-45.87, -67.50),
        'Ushuaia': (-54.80, -68.30),
    }
    for puerto, coords in puertos.items():
        folium.Marker(
            location=coords,
            popup=f'<b>{puerto}</b>',
            icon=folium.Icon(color='orange', icon='anchor', prefix='fa')
        ).add_to(m)
    
    # Mareas individuales como círculos
    for _, row in df.sample(80).iterrows():
        folium.CircleMarker(
            location=[row['lat_promedio'], row['lon_promedio']],
            radius=max(4, row['captura_ton'] / 50),
            color='white', weight=0.5,
            fill_color='#059669' if row['margen_pct'] > 45 else '#0099CC',
            fill_opacity=0.7,
            popup=(f"<b>{row['barco_nombre']}</b><br>"
                   f"Especie: {row['especie_principal']}<br>"
                   f"Captura: {row['captura_ton']:.0f} ton<br>"
                   f"Margen: {row['margen_pct']:.1f}%")
        ).add_to(m)
    
    m.save('mapa_actividad_flota.html')
    print('Mapa guardado en: mapa_actividad_flota.html')
    print('Abrilo en tu navegador para verlo interactivo.')
    display(m)

else:
    # Mapa estático con matplotlib
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.set_facecolor('#0A1628')
    fig.patch.set_facecolor('#0A1628')
    
    # Zonas de pesca como áreas
    zonas_viz = [
        ('Zona A - Norte PCA', -40.5, -58.0, UTNCELESTE),
        ('Zona B - Golfo San Jorge', -45.5, -61.5, PESCATEAL),
        ('Zona C - Banco Burdwood', -52.5, -60.0, ARIELBLUE),
        ('Zona D - Frente Malvinas', -44.0, -58.5, TECHGOLD),
        ('Zona E - Sur Patagónico', -48.0, -63.0, PESCAOCEAN),
    ]
    for nombre, lat, lon, color in zonas_viz:
        circle = plt.Circle((lon, lat), 1.5, color=color, alpha=0.12, linewidth=0)
        ax.add_patch(circle)
        ax.text(lon, lat + 1.7, nombre.split(' - ')[1], ha='center', fontsize=8.5,
                color=color, fontweight='bold')
    
    # Puntos de mareas
    captura_max = df['captura_ton'].max()
    scatter = ax.scatter(
        df['lon_promedio'], df['lat_promedio'],
        s=df['captura_ton'] / captura_max * 180,
        c=df['margen_pct'], cmap='RdYlGn',
        alpha=0.6, edgecolors='white', linewidth=0.3,
        vmin=20, vmax=70
    )
    
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label('Margen operativo (%)', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
    
    # Puertos
    puertos = [
        ('Mar del Plata', -57.53, -38.00),
        ('Rawson', -65.10, -43.30),
        ('Puerto Madryn', -65.03, -42.77),
        ('Comodoro Rivadavia', -67.50, -45.87),
        ('Ushuaia', -68.30, -54.80),
    ]
    for nombre, lon, lat in puertos:
        ax.plot(lon, lat, '^', markersize=10, color=TECHGOLD, zorder=5)
        ax.text(lon + 0.3, lat, nombre, fontsize=8, color=TECHGOLD, fontweight='bold')
    
    # Límite ZEE aproximado
    zee_lat = np.linspace(-34, -56, 50)
    zee_lon = -56 - (zee_lat + 34) * 0.3
    ax.plot(zee_lon, zee_lat, '--', color='#FF6B6B', alpha=0.5, linewidth=1.5,
            label='Límite ZEE (aprox.)')
    
    ax.set_xlim(-72, -52)
    ax.set_ylim(-57, -34)
    ax.set_xlabel('Longitud (°O)', color='white')
    ax.set_ylabel('Latitud (°S)', color='white')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#334155')
    ax.legend(fontsize=9, facecolor='#1E293B', labelcolor='white', edgecolor='#334155')
    ax.set_title('Mapa de Actividad de Flota — Plataforma Continental Argentina\n'
                 'Color = Margen operativo  |  Tamaño = Toneladas capturadas  |  Triángulos = Puertos base',
                 color='white', fontweight='bold')
    
    # Nota sobre folium
    ax.text(0.02, 0.02, 'TIP: instalar folium para versión interactiva (pip install folium)',
            transform=ax.transAxes, fontsize=8, color='#94A3B8',
            style='italic', verticalalignment='bottom')
    
    plt.tight_layout()
    plt.show()

---
## Parte 6 — Análisis por especie: composición de capturas

In [ ]:
# Composición por especie
df_especie = df.groupby('especie_principal').agg(
    captura_ton=('captura_ton', 'sum'),
    ingreso_usd=('ingreso_usd', 'sum'),
    n_mareas=('marea_id', 'count'),
    precio_medio=('precio_usd_ton', 'mean'),
    eficiencia_media=('eficiencia_ton_kl', 'mean'),
).reset_index().sort_values('ingreso_usd', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

COLORS_ESP = [UTNAZUL, PESCATEAL, TECHGOLD, PESCAOCEAN, ROJO]

# Torta de ingresos por especie
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    df_especie['ingreso_usd'],
    labels=df_especie['especie_principal'],
    colors=COLORS_ESP,
    autopct='%1.1f%%',
    pctdistance=0.82,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
ax.set_title('Composición de ingresos\npor especie', fontweight='bold', color=UTNAZUL)

# Barras: ton capturadas vs n_mareas
ax = axes[1]
ax2 = ax.twinx()
x = np.arange(len(df_especie))
bars = ax.bar(x, df_especie['captura_ton'], color=COLORS_ESP, alpha=0.8, width=0.5)
line = ax2.plot(x, df_especie['n_mareas'], 'D--', color=TECHGOLD,
                linewidth=2, markersize=8, markerfacecolor=TECHGOLD)
ax.set_xticks(x)
ax.set_xticklabels(
    [e.split()[0] for e in df_especie['especie_principal']],
    rotation=20
)
ax.set_ylabel('Toneladas capturadas', color=UTNAZUL)
ax2.set_ylabel('N° de mareas', color=TECHGOLD)
ax.set_title('Captura total vs N° de mareas\npor especie', fontweight='bold', color=UTNAZUL)

# Precio vs Eficiencia
ax = axes[2]
sc = ax.scatter(
    df_especie['precio_medio'],
    df_especie['eficiencia_media'],
    s=df_especie['ingreso_usd'] / 5000,
    c=COLORS_ESP, alpha=0.85,
    edgecolors='white', linewidth=1.5
)
for _, row in df_especie.iterrows():
    ax.annotate(
        row['especie_principal'].split()[0],
        (row['precio_medio'], row['eficiencia_media']),
        textcoords='offset points', xytext=(7, 5), fontsize=9,
        color=UTNAZUL, fontweight='bold'
    )
ax.set_xlabel('Precio promedio (USD/ton)')
ax.set_ylabel('Eficiencia (ton/kL)')
ax.set_title('Precio vs Eficiencia por especie\n(tamaño = ingreso total)',
             fontweight='bold', color=UTNAZUL)

plt.suptitle('Análisis por Especie — 6 meses de operaciones',
             fontsize=13, fontweight='bold', color=UTNAZUL)
plt.tight_layout()
plt.show()

---
## Parte 7 — Seguimiento de cuotas de captura

In [ ]:
# Simulamos cuotas asignadas para el semestre
cuotas = {
    'Merluza hubbsi': 18000,
    'Calamar illex': 25000,
    'Langostino patagónico': 8000,
    'Polaca': 6000,
    'Castañeta': 4000,
}

captura_real = df.groupby('especie_principal')['captura_ton'].sum().to_dict()

fig, ax = plt.subplots(figsize=(12, 5))

especies_ord = list(cuotas.keys())
cuotas_v = [cuotas[e] for e in especies_ord]
capturas_v = [captura_real.get(e, 0) for e in especies_ord]
pcts = [min(c/q*100, 110) for c, q in zip(capturas_v, cuotas_v)]

x = np.arange(len(especies_ord))
w = 0.38

b1 = ax.bar(x - w/2, cuotas_v, w, color=GRIS, alpha=0.4, label='Cuota asignada', zorder=2)
b2 = ax.bar(x + w/2, capturas_v, w,
            color=[PESCATEAL if p < 85 else TECHGOLD if p < 100 else ROJO for p in pcts],
            alpha=0.85, label='Captura real', zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(especies_ord, fontsize=11)
ax.set_ylabel('Toneladas')
ax.set_title('Seguimiento de Cuotas de Captura — Semestre Jul-Dic 2025\n'
             'Verde: OK | Dorado: cerca del límite | Rojo: límite superado',
             fontweight='bold', color=UTNAZUL)
ax.legend()
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Porcentaje de uso
for xi, (pct, cap, cuota) in enumerate(zip(pcts, capturas_v, cuotas_v)):
    color_txt = PESCATEAL if pct < 85 else TECHGOLD if pct < 100 else ROJO
    ax.text(xi + w/2, max(cap, cuota) + 200, f'{pct:.0f}%',
            ha='center', fontweight='bold', fontsize=11, color=color_txt)

plt.tight_layout()
plt.show()

print('Estado de cuotas al cierre del semestre:')
print(f'{"Especie":<28} {"Cuota":>8} {"Captura":>8} {"Uso %":>8} {"Estado"}')
print('-' * 65)
for esp in especies_ord:
    cuota = cuotas[esp]
    cap = captura_real.get(esp, 0)
    pct = cap / cuota * 100
    estado = 'OK' if pct < 85 else 'ATENCIÓN' if pct < 100 else 'EXCESO'
    print(f'{esp:<28} {cuota:>8,.0f} {cap:>8,.0f} {pct:>7.1f}% {estado}')

---
## Parte 8 — Exportación del reporte y herramientas de BI

### Python vs Power BI / Tableau

| Aspecto | Python (Plotly/Matplotlib) | Power BI / Tableau |
|---------|--------------------------|--------------------|
| **Flexibilidad** | Total — cualquier visualización programable | Limitada a los tipos de gráfico del producto |
| **Interactividad** | Plotly: sí (web); Matplotlib: no | Sí, nativa |
| **Curva de aprendizaje** | Mayor (código) | Menor (drag-and-drop) |
| **Actualización automática** | Sí (con scheduler/cron) | Sí (Power BI: con gateway) |
| **Costo** | Gratis (open source) | Power BI: USD 10/mes/user; Tableau: USD 75+ |
| **Integración con ML** | Nativa — mismo entorno | Requiere conector o R/Python embebido |
| **Ideal para** | Análisis técnico, reportes automatizados | Dashboards de gerencia, self-service BI |

**Recomendación para el sector pesquero:**
- **Operaciones y ML:** Python + Plotly
- **Reportes de gerencia:** Power BI conectado a la misma base de datos
- **Organismos reguladores:** Excel / PDF exportados automáticamente

In [ ]:
# Exportar reporte CSV para Power BI
reporte = df.groupby(['mes', 'barco_nombre', 'especie_principal', 'zona']).agg(
    n_mareas=('marea_id', 'count'),
    captura_ton=('captura_ton', 'sum'),
    ingreso_usd=('ingreso_usd', 'sum'),
    costo_combustible_usd=('costo_combustible_usd', 'sum'),
    eficiencia_media=('eficiencia_ton_kl', 'mean'),
    margen_medio=('margen_pct', 'mean'),
).reset_index()

# Exportar
reporte.to_csv('reporte_flota_powerbi.csv', index=False, encoding='utf-8-sig')

print(f'Reporte exportado: reporte_flota_powerbi.csv')
print(f'  Filas: {len(reporte)}')
print(f'  Columnas: {list(reporte.columns)}')
print()
print('Para conectar en Power BI:')
print('  1. Inicio → Obtener datos → Texto/CSV')
print('  2. Seleccionar reporte_flota_powerbi.csv')
print('  3. Transformar datos si es necesario (columna "mes" como fecha)')
print('  4. Crear visualizaciones con drag-and-drop')
print()
print('Nota: Si los datos vienen de una base SQL (PostgreSQL/MySQL),')
print('Power BI puede conectarse directamente con actualización automática.')
reporte.head(5)

---
## Parte 9 — Dashboard completo en una sola figura (para presentar)

In [ ]:
fig = plt.figure(figsize=(20, 12), facecolor='#F0F4F8')
gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35,
                      left=0.05, right=0.97, top=0.90, bottom=0.06)

# ─── HEADER ─────────────────────────────────────────────────────────────────
ax_header = fig.add_axes([0, 0.92, 1, 0.08])
ax_header.set_facecolor(UTNAZUL)
ax_header.text(0.5, 0.55, 'DASHBOARD OPERATIVO — FLOTA PESQUERA PATAGÓNICA',
               transform=ax_header.transAxes, ha='center', va='center',
               fontsize=16, fontweight='bold', color='white')
ax_header.text(0.5, 0.15, 'Período: Julio – Diciembre 2025  |  15 barcos  |  Datos actualizados: 31/12/2025',
               transform=ax_header.transAxes, ha='center', va='center',
               fontsize=10, color='#93C5FD')
ax_header.axis('off')

# ─── KPIs (fila 0) ──────────────────────────────────────────────────────────
kpi_data = [
    ('Captura Total', f"{df['captura_ton'].sum():,.0f} ton", UTNAZUL),
    ('Ingreso Total', f"USD {df['ingreso_usd'].sum()/1e6:.1f} M", PESCATEAL),
    ('Costo Gasoil', f"USD {df['costo_combustible_usd'].sum()/1e6:.1f} M", ROJO),
    ('Margen Promedio', f"{df['margen_pct'].mean():.1f}%", TECHGOLD),
]
for col, (titulo, valor, color) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor(color)
    ax.text(0.5, 0.60, valor, transform=ax.transAxes, ha='center', va='center',
            fontsize=20, fontweight='bold', color='white')
    ax.text(0.5, 0.20, titulo, transform=ax.transAxes, ha='center', va='center',
            fontsize=11, color='white')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('white'); spine.set_linewidth(2)

# ─── CAPTURA POR MES (fila 1, col 0-1) ──────────────────────────────────────
ax1 = fig.add_subplot(gs[1, :2])
ax1.bar(meses_labels, df_mes['captura_ton'],
        color=[UTNAZUL if v >= df_mes['captura_ton'].mean() else UTNCELESTE
               for v in df_mes['captura_ton']], alpha=0.85)
ax1.axhline(df_mes['captura_ton'].mean(), color=TECHGOLD, linestyle='--', lw=2)
ax1.set_title('Captura mensual total (ton)', fontweight='bold', color=UTNAZUL)
ax1.set_ylabel('Toneladas')
ax1.yaxis.grid(True, alpha=0.3); ax1.set_axisbelow(True)

# ─── TORTA ESPECIE (fila 1, col 2-3) ────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 2:])
ax2.pie(df_especie['ingreso_usd'], labels=df_especie['especie_principal'],
        colors=COLORS_ESP, autopct='%1.1f%%', pctdistance=0.82,
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax2.set_title('Ingresos por especie', fontweight='bold', color=UTNAZUL)

# ─── RANKING BARCOS (fila 2, col 0-1) ───────────────────────────────────────
ax3 = fig.add_subplot(gs[2, :2])
top10 = df_barco.head(10)
colors_top = [PESCATEAL if i < 3 else UTNCELESTE if i < 7 else GRIS
              for i in range(len(top10))]
ax3.barh(top10['barco_nombre'][::-1], top10['utilidad_usd'][::-1]/1000,
         color=colors_top[::-1], alpha=0.85)
ax3.set_xlabel('Utilidad (KUSD)')
ax3.set_title('Top 10 barcos por utilidad (6 meses)', fontweight='bold', color=UTNAZUL)
ax3.xaxis.grid(True, alpha=0.3); ax3.set_axisbelow(True)

# ─── EFICIENCIA POR MES (fila 2, col 2) ─────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 2])
ax4.plot(meses_labels, df_mes['eficiencia_media'], 'o-',
         color=ARIELBLUE, linewidth=2.5, markersize=8, markerfacecolor=TECHGOLD)
ax4.fill_between(range(6), df_mes['eficiencia_media'], alpha=0.15, color=ARIELBLUE)
ax4.set_title('Eficiencia (ton/kL)', fontweight='bold', color=UTNAZUL)
ax4.yaxis.grid(True, alpha=0.3); ax4.set_axisbelow(True)

# ─── MARGEN POR MES (fila 2, col 3) ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 3])
ax5.bar(meses_labels, df_mes['margen_medio'],
        color=[PESCATEAL if m > 45 else UTNCELESTE if m > 30 else ROJO
               for m in df_mes['margen_medio']], alpha=0.85)
ax5.set_title('Margen mensual (%)', fontweight='bold', color=UTNAZUL)
ax5.yaxis.grid(True, alpha=0.3); ax5.set_axisbelow(True)

plt.savefig('dashboard_flota_pesquera.png', dpi=150, bbox_inches='tight',
            facecolor='#F0F4F8')
print('Dashboard guardado: dashboard_flota_pesquera.png')
plt.show()

---
## Parte 10 — Reflexión y conexión con el curso

### ¿Qué tipos de dashboard necesita el sector pesquero?

| Audiencia | Dashboard | KPIs clave |
|-----------|-----------|------------|
| **Capitán de flota** | Operativo en tiempo real | Posición barcos, cuota disponible, clima |
| **Gerencia de planta** | Producción diaria | Ton procesadas, rendimiento, rechazos calidad |
| **Área comercial** | Exportaciones | Precios por mercado, volúmenes, certificaciones |
| **RRHH** | Flota embarcada | Dotación, rotación, horas embarcadas |
| **Organismo regulador** | Cumplimiento | Cuotas vs capturas, mareas vs vedas, informes VMS |
| **Investigación (INIDEP)** | Abundancia | Índices CPUE, distribución espacial, tendencias |

### Conexión con el resto del curso
- **Clase 4:** Los datos SST y AIS que exploramos ahí alimentan el mapa de actividad de hoy
- **Clase 6:** El modelo de predicción de zonas puede integrarse como un widget del dashboard
- **Clase 8:** El simulador de ahorro de combustible genera los KPIs que se muestran aquí
- **Clase 9 (hoy):** Todo converge — el dashboard es la interfaz que une todos los modelos

In [ ]:
print('RESUMEN DEL NOTEBOOK — Clase 9: Visualización y Dashboards')
print('=' * 62)
print()
print('Dataset: operaciones de 15 barcos durante 6 meses')
print(f'  Mareas analizadas: {len(df)}')
print(f'  Captura total:     {df["captura_ton"].sum():,.0f} ton')
print(f'  Ingreso total:     USD {df["ingreso_usd"].sum()/1e6:.1f} M')
print()
print('Visualizaciones creadas:')
print('  1. KPI cards del dashboard operativo')
print('  2. Series temporales de captura, eficiencia y margen')
print('  3. Ranking de barcos y scatter eficiencia vs margen')
print('  4. Mapa de actividad de flota (matplotlib / folium)')
print('  5. Análisis por especie (torta, barras, scatter)')
print('  6. Semáforo de cuotas de captura')
print('  7. Dashboard completo en una figura exportable')
print()
print('Archivos exportados:')
print('  - reporte_flota_powerbi.csv  → conectar con Power BI')
print('  - dashboard_flota_pesquera.png → presentar a gerencia')
print()
print('Materiales: github.com/PesquerosEnIA/curso-ia-produccion-pesquera')